In [ ]:
# this notebook conduct the following task:

# Use the sabdad antigen info  '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/'
# Use the sabdad antigen SurfaceIndex SI info /content/drive/MyDrive/@antibody/antigen_epitope/abdb/antigen_info_SI/
# Generate positive and negative patches datasets


In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
!pip install --upgrade --force-reinstall pandas
import pandas as pd

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import itertools
import math

In [ ]:
#@title Get their antigen info from tsv files
dir_name = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info/'

# Get a list of all .tsv files in the directory
tsv_files = [file for file in os.listdir(dir_name) if file.endswith('.tsv')]
print(len(tsv_files))


In [ ]:
tsv_files[:10]

In [ ]:
# Extract PDB IDs and select unique ones
unique_pdb_ids = set(file[:4] for file in tsv_files)  # First 4 characters are PDB ID

# Select one file per unique PDB ID
unique_selected_tsv_files = []
for pdb_id in unique_pdb_ids:
    for file in tsv_files:
        if file.startswith(pdb_id):
            unique_selected_tsv_files.append(file)
            break  # Only add the first file found for this PDB ID and then move to the next

len(unique_selected_tsv_files), unique_selected_tsv_files[:10]  # Show the number of unique files and the first 10 for brevity

In [ ]:
# Read the first matching .tsv file for each name in dat_files if they have more than 60 lines and combine them
combined_dfs = []

for tsv in unique_selected_tsv_files:
    file_path = os.path.join(dir_name, tsv)
    # Check the number of lines in the file before reading
    num_lines = sum(1 for line in open(file_path))
    if num_lines > 60:
        df = pd.read_csv(file_path, sep='\t')
        combined_dfs.append(df)

combined_df = pd.concat(combined_dfs, ignore_index=True)

In [ ]:
combined_df.head()

In [ ]:
print(len(combined_df["name"].unique()))

In [ ]:
# interacting info of antigen columns
ab_cols = ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N',
           'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']

# Add Status column
combined_df['Status'] = np.where(
    combined_df[ab_cols].sum(axis=1) == 0,
    'Non-Interacting',
    'Interacting'
)

# filter out non-Standard amino acid codes
standard_amino_acids = [
    'ALA', 'CYS', 'ASP', 'GLU', 'PHE',
    'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
    'MET', 'ASN', 'PRO', 'GLN', 'ARG',
    'SER', 'THR', 'VAL', 'TRP', 'TYR'
]

# Filter the dataframe
combined_df = combined_df[combined_df['AA'].isin(standard_amino_acids)]

In [ ]:
print(len(combined_df))

In [ ]:
#@title save df to csv
dir_name = '/content/drive/MyDrive/@antibody/antigen_epitope/sabdab/step2_prepare_datasets/'
combined_df.to_csv(os.path.join(dir_name, 'all_data.csv'))

In [ ]:
#@title merge the 'SI' information to all_data.csv
# from each TSV file in the directory /content/drive/MyDrive/@antibody/antigen_epitope/abdb/antigen_info_SI/ with the main all_data.csv file based on the 'name' and 'ResID' columns

import pandas as pd
import os

# Paths to the directories and files
tsv_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info_SI/"
all_data_path = "/content/drive/MyDrive/@antibody/antigen_epitope/sabdab/step2_prepare_datasets/all_data.csv"

# Load the main CSV file
all_data_df = pd.read_csv(all_data_path)

# Create a list to hold DataFrames of SI values
si_values_dfs = []

# Iterate over each TSV file in the directory and add to the list
for tsv_filename in os.listdir(tsv_dir):
    if tsv_filename.endswith('.tsv'):
        tsv_path = os.path.join(tsv_dir, tsv_filename)
        tsv_df = pd.read_csv(tsv_path, sep='\t')
        si_values_dfs.append(tsv_df[['name', 'ResID', 'SI']])

# Concatenate all DataFrames in the list
si_values_df = pd.concat(si_values_dfs, ignore_index=True)

# Merge the SI values into all_data_df
all_data_df = pd.merge(all_data_df, si_values_df, on=['name', 'ResID'], how='left')

# Save the updated DataFrame to a new CSV file with SI information included
updated_all_data_path = "/content/drive/MyDrive/@antibody/antigen_epitope/sabdab/step2_prepare_datasets/all_data_with_SI.csv"
all_data_df.to_csv(updated_all_data_path, index=False)

In [ ]:
all_data_df.head(10)

In [ ]:
unique_names_count = all_data_df['name'].nunique()
unique_names_count

In [ ]:
# Extract the first 4 characters from the 'name' column and count unique values
unique_names_4char_count = all_data_df['name'].str[:4].nunique()
unique_names_4char_count

In [ ]:
#@title read df from csv
dir_name = '/content/drive/MyDrive/@antibody/antigen_epitope/sabdab/step2_prepare_datasets/'
combined_df = pd.read_csv(os.path.join(dir_name, 'all_data_with_SI.csv'), index_col=0)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
combined_df.head(10)

In [ ]:
print(combined_df['name'].nunique())

In [ ]:
# Create the half-and-half violin plot
plt.figure(figsize=(36, 6))
sns.violinplot(data=combined_df, x='AA', y='SaSaRate', hue='Status', split=True)

plt.title('Amino Acid SASA Rates')
plt.xlabel('Types of Amino Acids')
plt.ylabel('SaSaRate')
plt.xticks(rotation=0)

# Optional: To keep the order of amino acids consistent, you can manually set the x-axis limits
plt.xlim(-0.5, len(combined_df['AA'].unique())-0.5)

plt.show()

In [ ]:
# Delete rows where SaSaRate is <2.0.
# filtered_combined_df = combined_df[combined_df['SaSaRate'] >= 0.2]  # set up the SaSaRate later
filtered_combined_df = combined_df

In [ ]:
#@title Extract the positive patch information from the data frame, and then convert patch info into 15*20 matrix
# see TEST for more details

# Create or overwrite cnn_positive_patch_3points.txt at the beginning
with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_positive_patch_3points.txt', 'w') as f:
    f.write('')  # Writing an empty string to ensure the file is empty

# One-hot encoding for amino acid types
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

# One-hot encoding for secondary structure
def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

# One-hot encoding for SaSaRate (Now 5 levels)
def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

# Combine 3-d one-hot encoded vector for secondary structure with 5-d one-hot encoded vector for SaSaRate
# Resulting in a 15-d vector
def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

def calculate_distance(residue1, residue2):
    x1, y1, z1 = residue1['SCx'], residue1['SCy'], residue1['SCz']
    x2, y2, z2 = residue2['SCx'], residue2['SCy'], residue2['SCz']
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

# Initialize an empty list to store all positive patches
positive_patches = []

# Loop through each group (protein)
for name, group in filtered_combined_df.groupby('name'):
    #interacting_residues = group[group['Status'] == 'Interacting']
    interacting_residues = group[(group['Status'] == 'Interacting') & (group['SaSaRate'] > 0.1)]
    patches = []

    for comb in itertools.combinations(interacting_residues.iterrows(), 3):
        index1, row1 = comb[0]
        index2, row2 = comb[1]
        index3, row3 = comb[2]

        d12 = calculate_distance(row1, row2)
        d23 = calculate_distance(row2, row3)
        d31 = calculate_distance(row3, row1)

        if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
            patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]

            patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                          [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                          [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]]

            patches.append({'indices': patch_indices, 'info': patch_info})

    # Output patches to a text file
    with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_positive_patch_3points.txt', 'a') as f:
        f.write(f"Patches for protein {name}:\n")
        for patch in patches:
            f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

    # Convert each patch into a 15*20 matrix and append it to positive_patches
    for patch in patches:
        patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
        for idx, aa_info in enumerate(patch['info']):
            aa = aa_info[0]
            ss = aa_info[1]
            sasa = aa_info[2]
            si = aa_info[3]  # Extract the SI value

            aa_encoded = encode_aa(aa)
            ss_encoded = encode_ss(ss)
            sasa_encoded = encode_sasa(sasa)

            combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

            for i in range(20):
                if aa_encoded[i] == 1:
                    patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value

        positive_patches.append(patch_matrix)

# Convert positive_patches to a NumPy array
positive_patches_3points = np.array(positive_patches)

# Save the NumPy array
save_dir = './drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/'
np.save(f'{save_dir}cnn_positive_patch_3points.npy', positive_patches_3points)

In [ ]:
len(positive_patches_3points)

In [ ]:
# Set print options for numpy to print the entire array neatly
np.set_printoptions(linewidth=np.inf, threshold=np.inf)
print(positive_patches_3points[1000])

In [ ]:
#@title Extract the negative patch information from the data frame
# randomly pick 500k from total negative patches

import math
import itertools
import numpy as np
import pandas as pd  # Assuming you use pandas for data frame operations

# Create or overwrite cnn_positive_patch_3points.txt at the beginning
with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_negative_patch_3points.txt', 'w') as f:
    f.write('')  # Writing an empty string to ensure the file is empty


# Define your functions as before but with slight modifications
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

# NumPy is more efficient for vectorized operations
def calculate_distance(residue1, residue2):
    coord1 = np.array([residue1['SCx'], residue1['SCy'], residue1['SCz']])
    coord2 = np.array([residue2['SCx'], residue2['SCy'], residue2['SCz']])
    return np.linalg.norm(coord1 - coord2)

patch_count = 0

# Pre-compute the total number of rows in the data frame for performance
total_rows = filtered_combined_df[filtered_combined_df['Status'] == 'Non-Interacting'].shape[0]

all_negative_patches = []

for name, group in filtered_combined_df.groupby('name'):
    non_interacting_residues = group[(group['Status'] == 'Non-Interacting' ) & (group['SaSaRate'] > 0.1)]
    patches = []

    # Calculate N for the current protein group
    N = int(200_000 * len(group) / total_rows)
    patches_found = 0  # Track how many patches we have found for this protein

    # Shuffle the rows for random selection
    shuffled_residues = non_interacting_residues.sample(frac=1).reset_index(drop=True)

    # one residue is not allowed in two different patches
    # used_residues = set()

    for comb in itertools.combinations(shuffled_residues.iterrows(), 3):
        if patches_found >= N:
            break  # We've found N patches for this protein, move to the next one

        index1, row1 = comb[0]
        index2, row2 = comb[1]
        index3, row3 = comb[2]

        # Skip this combination if any residue is already used in another patch
        # if index1 in used_residues or index2 in used_residues or index3 in used_residues:
        #    continue

        d12 = calculate_distance(row1, row2)
        d23 = calculate_distance(row2, row3)
        d31 = calculate_distance(row3, row1)

        if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
        #:  # Modified condition for SaSaRate

            patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                          [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                          [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]]
            patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
            patches.append({'indices': patch_indices, 'info': patch_info})

            # Mark these residues as used
            # used_residues.update([index1, index2, index3])

            patch_count += 1


    # Output patches to a text file
    with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_negative_patch_3points.txt', 'a') as f:
        f.write(f"Patches for protein {name}:\n")
        for patch in patches:
            f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

    for patch in patches:
        patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
        for idx, aa_info in enumerate(patch['info']):
            aa = aa_info[0]
            ss = aa_info[1]
            sasa = aa_info[2]
            si = aa_info[3]  # Extract the SI value

            aa_encoded = encode_aa(aa)
            ss_encoded = encode_ss(ss)
            sasa_encoded = encode_sasa(sasa)

            combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

            for i in range(20):
                if aa_encoded[i] == 1:
                    patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value
        all_negative_patches.append(patch_matrix)
        patches_found += 1
        print(f"Patches generated: {patches_found} for protein {name} with current N: {N}")

# Convert all_negative_patches to a NumPy array
all_negative_patches_3points = np.array(all_negative_patches)

# Save the NumPy array
save_dir = './drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/'
np.save(f'{save_dir}cnn_negative_patch_3points.npy', all_negative_patches_3points)

In [ ]:
all_negative_patches[0]